In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [32]:
import os
import json
import cv2
import torch
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [48]:
BASE_DIR = "/content/drive/MyDrive/Skylark/GCP_Assignment_Datasets"

TRAIN_ROOT = f"{BASE_DIR}/train_dataset"
TEST_ROOT = f"{BASE_DIR}/test_dataset"

LABEL_FILE = f"{TRAIN_ROOT}/curated_gcp_marks.json"

In [49]:
with open(LABEL_FILE) as f:
    labels = json.load(f)

print("Total training images:", len(labels))

Total training images: 1000


In [50]:
shape_to_idx = {
    "Cross":0,
    "Square":1
}

idx_to_shape = {v:k for k,v in shape_to_idx.items()}

In [51]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

In [52]:
class GCPDataset(Dataset):

    def __init__(self, labels, root):

        self.items = list(labels.items())
        self.root = root

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):

        path, data = self.items[idx]

        img_path = f"{self.root}/{path}"

        img = cv2.imread(img_path)

        h, w = img.shape[:2]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img_tensor = transform(img)

        # Normalize coordinates
        x = data["mark"]["x"] / w
        y = data["mark"]["y"] / h

        coords = torch.tensor([x,y], dtype=torch.float32)

        # Safe shape extraction
        shape_name = data.get("verified_shape","Cross")
        shape = shape_to_idx[shape_name]

        return img_tensor, coords, shape

In [53]:
dataset = GCPDataset(labels, TRAIN_ROOT)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

In [54]:
class GCPModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = models.resnet18(pretrained=True)

        features = self.backbone.fc.in_features

        self.backbone.fc = nn.Identity()

        self.coord_head = nn.Linear(features,2)
        self.class_head = nn.Linear(features,2)

    def forward(self,x):

        f = self.backbone(x)

        coords = self.coord_head(f)
        shape = self.class_head(f)

        return coords, shape

In [55]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GCPModel().to(device)

mse = nn.MSELoss()
ce = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [57]:
EPOCHS = 1

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for imgs, coords, shapes in tqdm(loader):

        imgs = imgs.to(device)
        coords = coords.to(device)
        shapes = shapes.to(device)

        pred_coords, pred_shapes = model(imgs)

        loss_coord = mse(pred_coords, coords)
        loss_shape = ce(pred_shapes, shapes)

        loss = loss_coord + loss_shape

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

100%|██████████| 63/63 [05:43<00:00,  5.46s/it]

Epoch 1/1 | Loss: 0.2379


In [58]:
test_images = []

for root, dirs, files in os.walk(TEST_ROOT):

    for file in files:

        if file.lower().endswith(".jpg"):

            full_path = os.path.join(root, file)

            test_images.append(full_path)

print("Total test images:", len(test_images))

Total test images: 300


In [59]:
model.eval()

predictions = {}

for img_path in tqdm(test_images):

    img = cv2.imread(img_path)

    h, w = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img_tensor = transform(img_rgb)

    img_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():

        coords, shape = model(img_tensor)

    # convert normalized coords back to pixels
    x = coords[0][0].item() * w
    y = coords[0][1].item() * h

    cls = torch.argmax(shape).item()

    # IMPORTANT: relative path from test_dataset
    rel_path = os.path.relpath(img_path, TEST_ROOT)

    predictions[rel_path] = {
        "mark": {
            "x": float(x),
            "y": float(y)
        },
        "verified_shape": idx_to_shape[cls]
    }

100%|██████████| 300/300 [04:06<00:00,  1.22it/s]


In [64]:
import json

save_path = "/content/drive/MyDrive/Skylark/predictions.json"

with open(save_path, "w") as f:
    json.dump(predictions, f, indent=4)

print("predictions.json saved to Skylark folder!")

predictions.json saved to Skylark folder!
